## Monte Carlo — introduction

> Rick Lupton, University of Bath (June 2026)

In [ ]:
import numpy as np
import scipy.stats

In [ ]:
# --- Presentation helpers (widgets + plotting only). Nothing conceptual here. ---
import matplotlib.pyplot as plt
from matplotlib import animation
import ipywidgets as widgets
from IPython.display import HTML, display


def _render_sequence(values):
    spans = [f'<span style="font-size:{14 + v * 6}px;color:hsl({int(v / 9 * 300)},70%,45%);'
             f'font-weight:700;margin:0 5px">{v}</span>' for v in values]
    return '<div style="min-height:70px;line-height:1.0">' + "".join(spans) + '</div>'


def rng_demo(seed):
    """A 'Draw' button that appends one random 0-9 to a growing, colour/size-coded row."""
    rng = np.random.default_rng(seed)
    drawn = []
    out = widgets.HTML(_render_sequence([]))
    btn = widgets.Button(description=f"Draw  (seed {seed})", button_style="primary")

    def draw(_):
        drawn.append(int(rng.integers(0, 10)))
        out.value = _render_sequence(drawn)

    btn.on_click(draw)
    display(widgets.VBox([btn, out]))


def animate_accumulation(x, mean, sd, lo, hi, n_frames=34):
    schedule = np.unique(np.round(np.geomspace(1, len(x), n_frames)).astype(int))
    grid = np.linspace(lo, hi, 300)
    pdf = scipy.stats.norm.pdf(grid, mean, sd)
    bins = np.linspace(lo, hi, 31)
    fig, (ax_rug, ax_hist) = plt.subplots(
        2, 1, figsize=(7.6, 5.0), sharex=True,
        gridspec_kw=dict(height_ratios=[1, 4], hspace=0.08))

    def draw(k):
        n = schedule[k]
        xn = x[:n]
        ax_rug.clear(); ax_hist.clear()
        ax_rug.plot(xn, np.zeros(n), "o", color="C0", alpha=0.30, ms=6, mfc="none")
        ax_rug.plot(xn[-1], 0, "o", color="C1", ms=9)
        ax_rug.set(ylim=(-1, 1), yticks=[])
        ax_rug.set_ylabel("draws", rotation=0, ha="right", va="center")
        ax_rug.set_title(f"n = {n} sample{'s' if n > 1 else ''}")
        ax_hist.hist(xn, bins=bins, density=True, color="C0", alpha=0.55)
        ax_hist.plot(grid, pdf, "k-", lw=2, label="true distribution")
        ax_hist.set(xlim=(lo, hi), ylim=(0, pdf.max() * 1.7))
        ax_hist.set_xlabel("mean vehicle lifetime  (years)")
        ax_hist.set_ylabel("probability density")
        ax_hist.legend(loc="upper right")

    anim = animation.FuncAnimation(fig, draw, frames=len(schedule), interval=280)
    plt.close(fig)
    return HTML(anim.to_jshtml(default_mode="once"))


def plot_transformation(x, y, func, lo, hi):
    fig = plt.figure(figsize=(7.8, 6.0))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 4], height_ratios=[4, 1],
                          wspace=0.06, hspace=0.06)
    ax_main = fig.add_subplot(gs[0, 1])
    ax_out = fig.add_subplot(gs[0, 0], sharey=ax_main)
    ax_in = fig.add_subplot(gs[1, 1], sharex=ax_main)
    grid = np.linspace(lo, hi, 300)
    ax_main.plot(grid, func(grid), "k-", lw=2)
    for xi in np.percentile(x, [5, 50, 95]):
        yi = func(xi)
        ax_main.plot([xi, xi], [0, yi], color="C1", lw=1, ls=":")
        ax_main.plot([lo, xi], [yi, yi], color="C1", lw=1, ls=":")
    ax_main.set(xlim=(lo, hi)); ax_main.tick_params(labelbottom=False, labelleft=False)
    ax_main.set_title("run each sample through the model,  y = f(L)")
    ax_in.hist(x, bins=30, density=True, color="C0", alpha=0.6)
    ax_in.invert_yaxis(); ax_in.set_yticks([])
    ax_in.set_xlabel("input:  mean lifetime L  (years)")
    ax_out.hist(y, bins=30, density=True, orientation="horizontal", color="C3", alpha=0.6)
    ax_out.invert_xaxis(); ax_out.set_xticks([])
    ax_out.set_ylabel("output:  annual outflow  (vehicles / yr)")


def plot_output_hist(y, pct):
    fig, ax = plt.subplots(figsize=(7.8, 4.0))
    ax.hist(y, bins=40, color="C3", alpha=0.6)
    ax.axvspan(pct[0], pct[-1], color="C1", alpha=0.15, label="5–95 %")
    ax.axvline(pct[2], color="C1", lw=2, ls="--", label="median")
    ax.set_xlabel("annual outflow  (vehicles / yr)"); ax.set_ylabel("number of samples")
    ax.set_title("distribution of the model output"); ax.legend(); fig.tight_layout()


def plot_convergence(y):
    ns = np.arange(1, len(y) + 1)
    med = np.array([np.percentile(y[:n], 50) for n in ns])
    p95 = np.array([np.percentile(y[:n], 95) for n in ns])
    fig, ax = plt.subplots(figsize=(7.8, 4.0))
    ax.plot(ns, med, color="C1", label="running median")
    ax.plot(ns, p95, color="C3", label="running 95th percentile")
    ax.axhline(np.percentile(y, 50), color="C1", ls=":", lw=1)
    ax.axhline(np.percentile(y, 95), color="C3", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_xlabel("number of samples used")
    ax.set_ylabel("estimated annual outflow")
    ax.set_title("estimates settle as we draw more samples"); ax.legend(); fig.tight_layout()

### 0. Random numbers and seeds

Same **seed** → same sequence. Two demos below share a seed; the third differs.

In [ ]:
rng_demo(seed=42)

In [ ]:
rng_demo(seed=42)

In [ ]:
rng_demo(seed=7)

### 1. An uncertain input, as a distribution

Mean vehicle lifetime: best guess **15 yr**, roughly ± 2 → a normal distribution.

In [ ]:
rng = np.random.default_rng(20260701)
rng.choice(["apples", "banana", "orange"], size=10)

In [ ]:
MEAN, SD = 15.0, 2.0

samples = rng.normal(MEAN, SD, size=2000)   # 2000 plausible lifetimes
samples[:5]

Draw many; the **histogram** fills in and converges to the true distribution (black).

In [ ]:
animate_accumulation(samples, MEAN, SD, lo=8, hi=22)

### 2. Propagate through a model

Run every sample through a model.

In [ ]:
STOCK_NOW = 30_000_000


def annual_outflow(lifetime):
    return STOCK_NOW / lifetime


outflow = annual_outflow(samples)   # one line: 2000 in, 2000 out
outflow[:5]

In [ ]:
plot_transformation(samples, outflow, annual_outflow, lo=8, hi=22)

### 3. Summarise with percentiles

Not mean ± sd (the output is skewed): read the **percentiles** off the samples.

In [ ]:
plot_output_hist(outflow, np.percentile(outflow, [5, 25, 50, 75, 95]))

In [ ]:
pct = np.percentile(outflow, [5, 25, 50, 75, 95])
print(f"median {pct[2]:,.0f}   |   90% range {pct[0]:,.0f} – {pct[4]:,.0f}  vehicles/yr")

### 4. How many samples?

Median settles fast; the 95th percentile (out in the tail) needs many more.
Error shrinks like `1 / sqrt(n)`.

In [ ]:
plot_convergence(outflow)

### Key ideas

- **Where does the uncertainty come from?** Missing knowledge (*epistemic*), not
  randomness → characterise it from data quality.
- **Is the model itself right?** Separate and often bigger from parameter values.
- **Are the inputs independent?** Real ones move together.
- **Uncertainty vs sensitivity analysis.** Which input drives the spread?
- **Answers about uncertainty depend on assumptions.**